# MATLAB PCD Analysis & Visualization
This notebook will help you deeply understand what data is contained inside your friend's MATLAB `.pcd` files! It will parse the raw points, count the objects, and render a colorful map of the scene so you can see exactly what the LiDAR scanner captured.

In [ ]:
# 1. Install required library
!pip install -q pypcd4 matplotlib numpy pandas

## 2. Load and Analyze the PCD File
Upload `frame_0002.pcd` to your Colab environment first!

In [ ]:
from pypcd4 import PointCloud
import numpy as np
import pandas as pd

# File to analyze
PCD_FILE = 'frame_0002.pcd'

# Mapping for the MATLAB Classes you provided
MATLAB_CLASSES = {
    0: 'Unlabeled / Background',
    1: 'Car',
    2: 'Truck',
    3: 'Bicycle',
    4: 'Pedestrian',
    5: 'Jersey Barrier',
    6: 'Guardrail'
}

print(f"Loading {PCD_FILE}...")
try:
    pc = PointCloud.from_path(PCD_FILE)
except Exception as e:
    print(f"ERROR: Could not find {PCD_FILE}. Please make sure you uploaded it to Colab!")
    raise e

# Extract coordinates
xyz = pc.numpy(['x', 'y', 'z'])

# Extract class labels (class_id)
try:
    labels = pc.pc_data['class_id'].astype(np.int32)
except ValueError:
    print("Warning: No 'class_id' field found in this PCD file! Defaulting to 0.")
    labels = np.zeros(xyz.shape[0], dtype=np.int32)

print(f"\nSuccessfully loaded {xyz.shape[0]:,} points from the point cloud!")

# Clean out NaN (Not-a-Number) values that might exist in raw LiDAR data
valid_mask = ~np.isnan(xyz).any(axis=1)
xyz = xyz[valid_mask]
labels = labels[valid_mask]

print(f"Retained {xyz.shape[0]:,} valid points after dropping NaNs.")

print("\n--- OBJECT STATISTICS ---")
# Count how many points belong to each class
unique_ids, counts = np.unique(labels, return_counts=True)

stats = []
for u_id, count in zip(unique_ids, counts):
    name = MATLAB_CLASSES.get(u_id, f"Unknown Class {u_id}")
    stats.append({"Class ID": u_id, "Object Name": name, "Point Count": count})

df = pd.DataFrame(stats)
display(df)


## 3. Visualize the Scene (Bird's Eye View)
Now let's render the point cloud from a top-down perspective, coloring each object differently so we can see the physical layout!

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Let's define some nice vivid colors for the MATLAB classes
MATLAB_COLORS = {
    0: [0.3, 0.3, 0.3],     # Background -> Dark Gray
    1: [0.0, 0.5, 1.0],     # Car -> Bright Blue
    2: [0.6, 0.2, 0.8],     # Truck -> Purple
    3: [0.0, 0.9, 0.9],     # Bicycle -> Cyan
    4: [1.0, 0.2, 0.2],     # Pedestrian -> Red
    5: [1.0, 0.6, 0.0],     # Jersey Barrier -> Orange
    6: [1.0, 1.0, 0.0]      # Guardrail -> Yellow
}

# We will map every single point to a color based on its label
point_colors = np.array([MATLAB_COLORS.get(l, [1, 1, 1]) for l in labels])

# Create the plot
plt.figure(figsize=(14, 12))

# Scatter plot using the X and Y coordinates (top-down view)
plt.scatter(xyz[:, 0], xyz[:, 1], c=point_colors, s=1.0, alpha=0.8)

# Create a nice legend
legend_patches = []
for u_id in unique_ids:
    color = MATLAB_COLORS.get(u_id, [1, 1, 1])
    name = MATLAB_CLASSES.get(u_id, f"Unknown ({u_id})")
    # Don't clutter legend with background if there are millions of points
    if name != 'Unlabeled / Background':
        patch = mpatches.Patch(color=color, label=name)
        legend_patches.append(patch)

if legend_patches:
    plt.legend(handles=legend_patches, loc='upper right', title="Detected Objects", framealpha=0.9)

plt.title(f"LiDAR Scene Visualization ({PCD_FILE})", fontsize=18)
plt.xlabel("X Coordinate (meters)")
plt.ylabel("Y Coordinate (meters)")
plt.axis('equal')
plt.gca().set_facecolor('black')
plt.show()
